# Cerința 7 — Spark Structured Streaming + Inferență în Timp Real

### Scenariu

Simulăm un **flux de date live de la un meci de fotbal**: evenimentele (tentative de șut) sosesc în timp real și sistemul nostru aplică **modelul ML antrenat** (pipeline-ul de la Cerința 4) pentru a prezice probabilitatea de gol pentru fiecare șut, în timp ce produce statistici agregate în ferestre de timp.

### Arhitectura pipeline-ului de streaming

```
Fișiere CSV simulate   →   Spark readStream   →   Preprocesare   →   ML inference
(meci live events)          (trigger 5s)          (fillna, cast)     (pipeline model)
                                                                            ↓
                                                                     writeStream
                                                                     (console output)
```

### Abordarea: Structured Streaming din fișiere

Simulăm stream-ul prin scrierea treptată a fișierelor CSV într-un director de intrare — Spark monitorizează directorul și procesează fișierele noi ca micro-batch-uri.


In [1]:
import os
# Notebook-urile stau în subfolderul notebooks/; ne asigurăm că lucrăm din rădăcina proiectului
# (unde se află data/, models/, plots/), astfel încât toate căile relative să funcționeze.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')
# Directoarele de output trebuie să existe înainte de orice savefig()/save()
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Working directory:', os.getcwd())

Working directory: /Users/stefan/Documents/football-events-analysis


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    current_timestamp, window, when, lit
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType
)
from pyspark.ml import PipelineModel
import pandas as pd
import numpy as np
import time
import os
import shutil
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('FootballStreaming') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.sql.streaming.schemaInference', 'true') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 20:08:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


## 7.1 Generarea datelor simulate (stream de intrare)

Împărțim dataset-ul real în "batch-uri" care simulează evenimentele unui meci ce sosesc pe rând — câte unul la fiecare câteva secunde.


In [3]:
# Directoare pentru stream
STREAM_INPUT_DIR  = 'data/stream_input'
STREAM_OUTPUT_DIR = 'data/stream_output'
CHECKPOINT_DIR    = 'data/stream_checkpoint'

for d in [STREAM_INPUT_DIR, STREAM_OUTPUT_DIR, CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Citim câteva sute de șuturi reale ca bază pentru simulare
events_real = spark.read.csv('data/events.csv', header=True, inferSchema=True, nullValue='NA') \
    .filter(
        (col('event_type') == 1) &
        col('is_goal').isNotNull() &
        col('location').isNotNull() &
        col('bodypart').isNotNull() &
        col('situation').isNotNull()
    ) \
    .fillna({'assist_method': 0, 'fast_break': 0, 'shot_place': 0, 'side': 1}) \
    .select('id_odsp', 'time', 'event_team', 'opponent',
            'is_goal', 'location', 'bodypart', 'situation',
            'assist_method', 'fast_break', 'shot_place', 'side') \
    .limit(500)

# Convertim în pandas pentru a scrie batch-uri simulate
events_pd = events_real.toPandas()
print(f'Date simulate: {len(events_pd)} tentative de șut')

# Împărțim în 5 batch-uri de ~100 rânduri fiecare
BATCH_SIZE = 100
batches = [events_pd.iloc[i:i+BATCH_SIZE] for i in range(0, len(events_pd), BATCH_SIZE)]
print(f'Batches pregătite: {len(batches)} × ~{BATCH_SIZE} rânduri')

Date simulate: 500 tentative de șut
Batches pregătite: 5 × ~100 rânduri


## 7.2 Definirea schemei și configurarea stream-ului de intrare

Definim explicit schema pentru a nu pierde timp la inferare și pentru a asigura consistența tipurilor.


In [4]:
stream_schema = StructType([
    StructField('id_odsp',      StringType(),  True),
    StructField('time',         DoubleType(),  True),
    StructField('event_team',   StringType(),  True),
    StructField('opponent',     StringType(),  True),
    StructField('is_goal',      IntegerType(), True),
    StructField('location',     IntegerType(), True),
    StructField('bodypart',     IntegerType(), True),
    StructField('situation',    IntegerType(), True),
    StructField('assist_method',IntegerType(), True),
    StructField('fast_break',   IntegerType(), True),
    StructField('shot_place',   IntegerType(), True),
    StructField('side',         IntegerType(), True)
])

# Scriem primul batch pentru a inițializa stream-ul
batch0_path = os.path.join(STREAM_INPUT_DIR, 'batch_0.csv')
batches[0].to_csv(batch0_path, index=False)
print(f'Batch 0 scris: {batch0_path} ({len(batches[0])} rânduri)')

# Citim stream-ul din director
stream_df = spark.readStream \
    .schema(stream_schema) \
    .option('header', 'true') \
    .option('maxFilesPerTrigger', 1) \
    .csv(STREAM_INPUT_DIR)

print(f'Stream isStreaming: {stream_df.isStreaming}')

Batch 0 scris: data/stream_input/batch_0.csv (100 rânduri)
Stream isStreaming: True


## 7.3 Aplicarea modelului ML în timp real

Aplicăm **pipeline-ul antrenat** (de la Cerința 4) pe fiecare micro-batch de date primite în stream.


In [5]:
# Încărcăm pipeline-ul ML antrenat anterior (Cerința 4)
# Dacă nu există, antrenăm un model simplu
pipeline_path = 'models/rf_pipeline_model'

if os.path.exists(pipeline_path):
    pipeline_model = PipelineModel.load(pipeline_path)
    print(f'Pipeline MLlib încărcat din: {pipeline_path}')
    
    # Aplicăm modelul pe stream
    # NOTĂ: VectorAssembler necesită coloanele corecte
    # Pipeline-ul din cerința 4 folosea și coloane inginerited (is_central_shot etc.)
    # Facem o preprocesare minimă compatibilă
    from pyspark.sql.functions import floor
    stream_enriched = stream_df \
        .withColumn('is_central_shot',
                    when(col('location').isin(3, 5, 10, 12, 13, 14), 1).otherwise(0)) \
        .withColumn('time_phase',
                    when(col('time') <= 45, 1).when(col('time') <= 70, 2).otherwise(3)) \
        .withColumn('is_set_piece',
                    when(col('situation').isin(2, 3, 4), 1).otherwise(0))

    stream_with_predictions = pipeline_model.transform(stream_enriched)
    
    use_ml_predictions = True
    print('Inferență ML activată pe stream.')
else:
    # Fallback: rulăm stream-ul fără inferență ML
    stream_with_predictions = stream_df
    use_ml_predictions = False
    print('ATENȚIE: Pipeline ML negăsit. Rulăm stream-ul fără inferență ML.')
    print('Rulați mai întâi Cerința 4 pentru a genera modelul.')

Pipeline MLlib încărcat din: models/rf_pipeline_model
Inferență ML activată pe stream.


## 7.4 Query 1 — Statistici în timp real pe echipă

Calculăm per fiecare echipă: numărul de șuturi, goluri marcate și rata de conversie curentă.


In [6]:
# Statistici agregate pe echipă
team_stats_stream = stream_df \
    .groupBy('event_team') \
    .agg(
        count('*').alias('suturi'),
        spark_sum('is_goal').alias('goluri')
    ) \
    .withColumn(
        'rata_conversie_pct',
        spark_round(col('goluri') * 100.0 / col('suturi'), 1)
    ) \
    .orderBy('goluri', ascending=False)

# Query Streaming cu outputMode 'complete' (pentru agregări)
query_team_stats = team_stats_stream.writeStream \
    .outputMode('complete') \
    .format('console') \
    .option('truncate', False) \
    .option('numRows', 10) \
    .trigger(processingTime='5 seconds') \
    .start()

print('Query 1 pornit: statistici per echipă (outputMode=complete)')
print('Adăugăm batch-uri noi la fiecare 6 secunde...')

# Simulăm sosirea datelor noi
for i, batch in enumerate(batches[1:], start=1):
    time.sleep(6)
    batch_path = os.path.join(STREAM_INPUT_DIR, f'batch_{i}.csv')
    batch.to_csv(batch_path, index=False)
    print(f'  Batch {i} adăugat: {len(batch)} rânduri noi')

# Așteptăm ultimul trigger
time.sleep(8)
query_team_stats.stop()
print('Query 1 oprit.')

Query 1 pornit: statistici per echipă (outputMode=complete)
Adăugăm batch-uri noi la fiecare 6 secunde...


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+------+------+------------------+
|event_team         |suturi|goluri|rata_conversie_pct|
+-------------------+------+------+------------------+
|Borussia Dortmund  |16    |3     |18.8              |
|SC Freiburg        |9     |2     |22.2              |
|Werder Bremen      |19    |2     |10.5              |
|FC Augsburg        |11    |2     |18.2              |
|Lorient            |9     |1     |11.1              |
|Hamburg SV         |8     |1     |12.5              |
|Valenciennes       |4     |0     |0.0               |
|Paris Saint-Germain|12    |0     |0.0               |
|Caen               |3     |0     |0.0               |
|Kaiserslautern     |9     |0     |0.0               |
+-------------------+------+------+------------------+



  Batch 1 adăugat: 100 rânduri noi


-------------------------------------------
Batch: 1
-------------------------------------------
+---------------------+------+------+------------------+
|event_team           |suturi|goluri|rata_conversie_pct|
+---------------------+------+------+------------------+
|Borussia Dortmund    |16    |3     |18.8              |
|SC Freiburg          |9     |2     |22.2              |
|Werder Bremen        |19    |2     |10.5              |
|Evian Thonon Gaillard|9     |2     |22.2              |
|Brest                |23    |2     |8.7               |
|Toulouse             |13    |2     |15.4              |
|FC Augsburg          |11    |2     |18.2              |
|Nice                 |3     |1     |33.3              |
|Nurnberg             |13    |1     |7.7               |
|Caen                 |11    |1     |9.1               |
+---------------------+------+------+------------------+
only showing top 10 rows


  Batch 2 adăugat: 100 rânduri noi


...

  Batch 3 adăugat: 100 rânduri noi


-------------------------------------------
Batch: 3
-------------------------------------------
+---------------------+------+------+------------------+
|event_team           |suturi|goluri|rata_conversie_pct|
+---------------------+------+------+------------------+
|Borussia Dortmund    |16    |3     |18.8              |
|Lyon                 |22    |3     |13.6              |
|VfB Stuttgart        |12    |3     |25.0              |
|VfL Wolfsburg        |18    |3     |16.7              |
|Montpellier          |13    |3     |23.1              |
|Werder Bremen        |19    |2     |10.5              |
|SC Freiburg          |9     |2     |22.2              |
|Evian Thonon Gaillard|9     |2     |22.2              |
|Brest                |23    |2     |8.7               |
|Sochaux              |8     |2     |25.0              |
+---------------------+------+------+------------------+
only showing top 10 rows


  Batch 4 adăugat: 100 rânduri noi


-------------------------------------------
Batch: 4
-------------------------------------------
+---------------------+------+------+------------------+
|event_team           |suturi|goluri|rata_conversie_pct|
+---------------------+------+------+------------------+
|Stade Rennes         |16    |5     |31.3              |
|Borussia Dortmund    |16    |3     |18.8              |
|Lyon                 |22    |3     |13.6              |
|VfB Stuttgart        |12    |3     |25.0              |
|VfL Wolfsburg        |18    |3     |16.7              |
|Montpellier          |13    |3     |23.1              |
|Mainz                |23    |2     |8.7               |
|Werder Bremen        |19    |2     |10.5              |
|SC Freiburg          |9     |2     |22.2              |
|Evian Thonon Gaillard|9     |2     |22.2              |
+---------------------+------+------+------------------+
only showing top 10 rows


Query 1 oprit.


## 7.5 Query 2 — Predicții individuale pe șuturi

Dacă modelul ML este disponibil, afișăm pentru fiecare șut nou: echipa, minutul, și probabilitatea predicției.


In [7]:
# Resetăm directorul de intrare
shutil.rmtree(STREAM_INPUT_DIR)
os.makedirs(STREAM_INPUT_DIR)
shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR)

# Re-citim stream-ul
stream_df2 = spark.readStream \
    .schema(stream_schema) \
    .option('header', 'true') \
    .option('maxFilesPerTrigger', 1) \
    .csv(STREAM_INPUT_DIR)

if use_ml_predictions:
    stream_enriched2 = stream_df2 \
        .withColumn('is_central_shot',
                    when(col('location').isin(3, 5, 10, 12, 13, 14), 1).otherwise(0)) \
        .withColumn('time_phase',
                    when(col('time') <= 45, 1).when(col('time') <= 70, 2).otherwise(3)) \
        .withColumn('is_set_piece',
                    when(col('situation').isin(2, 3, 4), 1).otherwise(0))

    predictions_stream = pipeline_model.transform(stream_enriched2)

    from pyspark.sql.functions import round as spark_round2
    output_stream = predictions_stream.select(
        'event_team',
        col('time').cast('int').alias('minut'),
        'location',
        'is_goal',
        'prediction',
    )
else:
    # Fără ML — afișăm datele brute
    output_stream = stream_df2.select(
        'event_team',
        col('time').cast('int').alias('minut'),
        'location',
        'bodypart',
        'is_goal'
    )

# Query cu outputMode 'append'
query_predictions = output_stream.writeStream \
    .outputMode('append') \
    .format('console') \
    .option('truncate', False) \
    .option('numRows', 20) \
    .trigger(processingTime='5 seconds') \
    .option('checkpointLocation', CHECKPOINT_DIR) \
    .start()

print('Query 2 pornit: predicții individuale pe șuturi (outputMode=append)')

# Adăugăm primele 3 batch-uri
for i in range(3):
    batches[i].to_csv(os.path.join(STREAM_INPUT_DIR, f'batch_{i}.csv'), index=False)
    print(f'  Batch {i} adăugat')
    time.sleep(7)

time.sleep(5)
query_predictions.stop()
print('Query 2 oprit.')

Query 2 pornit: predicții individuale pe șuturi (outputMode=append)
  Batch 0 adăugat


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+-----+--------+-------+----------+
|event_team       |minut|location|is_goal|prediction|
+-----------------+-----+--------+-------+----------+
|Hamburg SV       |2    |9       |0      |0.0       |
|Borussia Dortmund|14   |15      |0      |0.0       |
|Borussia Dortmund|17   |9       |1      |0.0       |
|Borussia Dortmund|19   |15      |0      |0.0       |
|Hamburg SV       |20   |15      |0      |0.0       |
|Borussia Dortmund|25   |3       |0      |0.0       |
|Borussia Dortmund|25   |15      |0      |0.0       |
|Borussia Dortmund|26   |3       |0      |0.0       |
|Borussia Dortmund|28   |9       |0      |0.0       |
|Borussia Dortmund|29   |3       |1      |0.0       |
|Borussia Dortmund|36   |11      |0      |0.0       |
|Hamburg SV       |38   |15      |0      |0.0       |
|Borussia Dortmund|39   |9       |0      |0.0       |
|Borussia Dortmund|45   |11      |0    

  Batch 1 adăugat


-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+-----+--------+-------+----------+
|event_team   |minut|location|is_goal|prediction|
+-------------+-----+--------+-------+----------+
|Caen         |33   |15      |0      |0.0       |
|Caen         |33   |3       |0      |0.0       |
|Caen         |34   |15      |1      |0.0       |
|Valenciennes |37   |3       |0      |0.0       |
|Caen         |40   |15      |0      |0.0       |
|Valenciennes |43   |15      |0      |0.0       |
|Valenciennes |43   |12      |0      |0.0       |
|Valenciennes |45   |3       |0      |0.0       |
|Valenciennes |48   |15      |0      |0.0       |
|Caen         |52   |15      |0      |0.0       |
|Caen         |54   |3       |0      |0.0       |
|Valenciennes |67   |9       |0      |0.0       |
|Caen         |87   |15      |0      |0.0       |
|Valenciennes |88   |11      |0      |0.0       |
|Valenciennes |88   |3       |0      |0.0       |
|Ca

  Batch 2 adăugat


-------------------------------------------
Batch: 2
-------------------------------------------
+----------+-----+--------+-------+----------+
|event_team|minut|location|is_goal|prediction|
+----------+-----+--------+-------+----------+
|Nice      |13   |9       |0      |0.0       |
|Nice      |13   |15      |0      |0.0       |
|Lyon      |14   |15      |0      |0.0       |
|Nice      |15   |17      |0      |0.0       |
|Lyon      |16   |3       |0      |0.0       |
|Nice      |23   |3       |0      |0.0       |
|Lyon      |25   |6       |0      |0.0       |
|Nice      |27   |15      |0      |0.0       |
|Lyon      |29   |17      |0      |0.0       |
|Lyon      |31   |15      |0      |0.0       |
|Lyon      |32   |15      |0      |0.0       |
|Lyon      |32   |13      |0      |0.0       |
|Lyon      |33   |13      |1      |1.0       |
|Lyon      |36   |15      |0      |0.0       |
|Lyon      |36   |9       |0      |0.0       |
|Lyon      |36   |3       |0      |0.0       |
|Nice     

Query 2 oprit.


## 7.6 Query 3 — Statistici finale agregate (batch mode)

Procesăm toate datele simulate ca un job batch final pentru a valida că stream-ul a prelucrat corect toate evenimentele.


In [8]:
# Scriem toate batch-urile simulate
all_batches_path = 'data/stream_all_batches.csv'
events_pd.to_csv(all_batches_path, index=False)

# Procesare batch finală pentru validare
final_df = spark.read.csv(all_batches_path, header=True, schema=stream_schema)

print('=== Statistici finale după procesarea stream-ului ===')
final_df.createOrReplaceTempView('stream_final')

spark.sql("""
    SELECT
        event_team,
        COUNT(*)                              AS total_suturi,
        SUM(is_goal)                          AS goluri_marcate,
        ROUND(SUM(is_goal)*100.0/COUNT(*), 1) AS rata_conv_pct,
        ROUND(AVG(time), 1)                   AS min_mediu_sut
    FROM stream_final
    GROUP BY event_team
    ORDER BY goluri_marcate DESC
    LIMIT 10
""").show(truncate=False)

# Curățare
for d in [STREAM_INPUT_DIR, STREAM_OUTPUT_DIR, CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)

spark.stop()
print('\nSesiune Spark oprită. Streaming demonstrat cu succes.')

=== Statistici finale după procesarea stream-ului ===


+-----------------+------------+--------------+-------------+-------------+
|event_team       |total_suturi|goluri_marcate|rata_conv_pct|min_mediu_sut|
+-----------------+------------+--------------+-------------+-------------+
|Stade Rennes     |16          |5             |31.3         |38.7         |
|Lyon             |22          |3             |13.6         |46.0         |
|Borussia Dortmund|16          |3             |18.8         |38.9         |
|VfL Wolfsburg    |18          |3             |16.7         |48.8         |
|VfB Stuttgart    |12          |3             |25.0         |52.8         |
|Montpellier      |13          |3             |23.1         |52.5         |
|Toulouse         |13          |2             |15.4         |58.1         |
|Brest            |23          |2             |8.7          |52.2         |
|Mainz            |23          |2             |8.7          |46.7         |
|Hannover 96      |7           |2             |28.6         |38.6         |
+-----------


Sesiune Spark oprită. Streaming demonstrat cu succes.


## Rezumat — Ce am demonstrat

| Aspect | Implementare |
|---|---|
| **Sursă stream** | Director de fișiere CSV (simulate cu `maxFilesPerTrigger=1`) |
| **Schema explicită** | `StructType` cu tipuri precise pentru toate coloanele |
| **Trigger** | `processingTime='5 seconds'` — micro-batch la fiecare 5s |
| **Output modes** | `complete` (pentru agregări) și `append` (pentru predicții individuale) |
| **Inferență ML** | Pipeline MLlib antrenat offline, aplicat pe fiecare micro-batch |
| **Checkpointing** | `checkpointLocation` — toleranță la defecte și restart |
| **Agregări** | `groupBy + agg` pe date de streaming (statistici per echipă) |
